In [35]:
#Face detector downloaded here:
#https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite

In [1]:
# !pip install ipykernel
# import sys
# !{sys.executable} -m ipykernel install --user --name posterbn --display-name "Python (posterbn)"
# !pip install numpy opencv-python pillow scipy scikit-image scikit-learn
# !pip install torch torchvision torchaudio
# !pip install ultralytics mediapipe easyocr open_clip_torch
# !pip install open-nsfw2

In [2]:
import sys
print(sys.executable)

C:\Users\Asher\anaconda3\envs\posterbn\python.exe


In [3]:
# import sys
# !{sys.executable} -m pip install numpy opencv-python pillow scipy scikit-image scikit-learn
# !{sys.executable} -m pip install torch torchvision torchaudio
# !{sys.executable} -m pip install ultralytics mediapipe easyocr open_clip_torch

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

True
NVIDIA GeForce RTX 3090 Ti


In [11]:
"""
Poster Node Extraction (expanded + partner-requested nodes)

External models/libs used (all local):
- Ultralytics YOLOv8 (object detection): https://docs.ultralytics.com/quickstart/  (pip: ultralytics)
- OpenCLIP (CLIP embeddings + zero-shot prompts): https://github.com/mlfoundations/open_clip
- MediaPipe Face Detector (faces): https://ai.google.dev/edge/mediapipe/solutions/vision/face_detector/python
- EasyOCR (OCR): https://github.com/JaidedAI/EasyOCR  and demo https://www.jaided.ai/easyocr/
Optional (local NSFW scoring):
- open_nsfw2: https://github.com/bhky/opennsfw2  (pip: open-nsfw2)

Install (pick what you need):
  pip install opencv-python pillow numpy scikit-image scikit-learn
  pip install ultralytics
  pip install open_clip_torch torch torchvision --index-url https://download.pytorch.org/whl/cu121   # choose your CUDA/CPU
  pip install mediapipe
  pip install easyocr
  pip install open-nsfw2   # optional

Notes:
- Some libs/models may auto-download weights on first run (free, not pay-per-use). If you need fully-offline:
  pre-download weights and point to local paths (YOLO weights, OpenCLIP cache, mediapipe face_detector.tflite, etc.).
"""

from __future__ import annotations

from typing import Dict, Any, List, Tuple, Optional
import math
import numpy as np
from PIL import Image
import cv2
import easyocr

from skimage.feature import local_binary_pattern
from sklearn.cluster import KMeans

In [42]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
from PIL import Image
import cv2
from skimage.feature import local_binary_pattern
from sklearn.cluster import KMeans

# ----------------------------
# Config
# ----------------------------
MAX_SIDE = 1024
FACE_MODEL_PATH = Path("./face_detector.tflite")
YOLO_WEIGHTS = "yolov8n.pt"

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "laion2b_s34b_b79k"

CLIP_PROMPTS = {
    "photographic": "a photographic movie poster",
    "illustrated": "an illustrated movie poster",
    "dark_moody": "a dark moody movie poster",
    "bright_colorful": "a bright colorful movie poster",
    "minimalist": "a minimalist movie poster",
    "crowded": "a crowded movie poster with many elements",
    "romantic": "a romantic movie poster",
    "horror": "a horror movie poster",
    "comedy": "a comedy movie poster",
    "action": "an action movie poster",
    "retro": "a retro vintage movie poster",
    "family_friendly": "a family-friendly movie poster",
    "serious_drama": "a serious drama movie poster",
    "sci_fi": "a science fiction movie poster",
    "historical": "a historical period movie poster",
    "explosion": "a movie poster with an explosion",
    "weapons": "a movie poster featuring weapons",
    "gun": "a movie poster featuring a gun",
    "vehicle": "a movie poster featuring a vehicle",
    "car_chase": "a movie poster with a car chase",
    "face_profile": "a movie poster with a face in profile",
    "face_obscured": "a movie poster with an obscured face",
    "masked_face": "a movie poster with a masked face",
    "serif_type": "a movie poster with serif typography",
    "sans_type": "a movie poster with sans-serif typography",
    "script_type": "a movie poster with script handwriting typography",
    "condensed_bold_type": "a movie poster with bold condensed typography",
}

# ----------------------------
# Global caches
# ----------------------------
_TORCH = None
_DEVICE = None

_YOLO_MODEL = None
_EASYOCR_READER = None

_MP = None
_MP_PYTHON = None
_MP_VISION = None
_FACE_DETECTOR = None
_FACE_DETECTOR_ERROR = None

_OPEN_CLIP = None
_CLIP_MODEL = None
_CLIP_PREPROCESS = None
_CLIP_TOKENIZER = None
_CLIP_TEXT_FEATURES = None

_NSFW2 = None
_NSFW2_ERROR = None


# ----------------------------
# Core helpers
# ----------------------------
def get_torch():
    global _TORCH
    if _TORCH is None:
        import torch
        _TORCH = torch
    return _TORCH


def get_device() -> str:
    global _DEVICE
    if _DEVICE is None:
        torch = get_torch()
        _DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    return _DEVICE


def load_rgb(path: str, max_side: int = MAX_SIDE) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return np.array(img, dtype=np.uint8)


def rgb_to_hsv(rgb: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)


def luminance(rgb: np.ndarray) -> np.ndarray:
    r = rgb[..., 0].astype(np.float32)
    g = rgb[..., 1].astype(np.float32)
    b = rgb[..., 2].astype(np.float32)
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def safe_div(a: float, b: float) -> float:
    return float(a) / float(b) if b != 0 else 0.0


def clamp01(x: float) -> float:
    return float(max(0.0, min(1.0, x)))


# ----------------------------
# Heavy model getters
# ----------------------------
def get_yolo_model():
    global _YOLO_MODEL
    if _YOLO_MODEL is None:
        from ultralytics import YOLO
        _YOLO_MODEL = YOLO(YOLO_WEIGHTS)
    return _YOLO_MODEL


def get_easyocr_reader():
    global _EASYOCR_READER
    if _EASYOCR_READER is None:
        import easyocr
        torch = get_torch()
        _EASYOCR_READER = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
    return _EASYOCR_READER


def get_mediapipe_face_detector():
    global _MP, _MP_PYTHON, _MP_VISION, _FACE_DETECTOR, _FACE_DETECTOR_ERROR

    if _FACE_DETECTOR is not None:
        return _FACE_DETECTOR, None

    if _FACE_DETECTOR_ERROR is not None:
        return None, _FACE_DETECTOR_ERROR

    try:
        import mediapipe as mp
        from mediapipe.tasks import python
        from mediapipe.tasks.python import vision

        _MP = mp
        _MP_PYTHON = python
        _MP_VISION = vision
    except Exception as e:
        _FACE_DETECTOR_ERROR = f"mediapipe not available: {e}"
        return None, _FACE_DETECTOR_ERROR

    if not FACE_MODEL_PATH.exists():
        _FACE_DETECTOR_ERROR = f"face detector model not found at {FACE_MODEL_PATH.resolve()}"
        return None, _FACE_DETECTOR_ERROR

    try:
        base_options = _MP_PYTHON.BaseOptions(model_asset_path=str(FACE_MODEL_PATH))
        options = _MP_VISION.FaceDetectorOptions(base_options=base_options)
        _FACE_DETECTOR = _MP_VISION.FaceDetector.create_from_options(options)
        return _FACE_DETECTOR, None
    except Exception as e:
        _FACE_DETECTOR_ERROR = f"mediapipe face model issue: {e}"
        return None, _FACE_DETECTOR_ERROR


def get_clip_components():
    global _OPEN_CLIP, _CLIP_MODEL, _CLIP_PREPROCESS, _CLIP_TOKENIZER, _CLIP_TEXT_FEATURES

    if _CLIP_MODEL is not None:
        return _CLIP_MODEL, _CLIP_PREPROCESS, _CLIP_TOKENIZER, _CLIP_TEXT_FEATURES

    torch = get_torch()
    import open_clip

    device = get_device()

    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME,
        pretrained=CLIP_PRETRAINED,
    )
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)

    prompt_texts = list(CLIP_PROMPTS.values())
    text_tokens = tokenizer(prompt_texts).to(device)

    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    _OPEN_CLIP = open_clip
    _CLIP_MODEL = model
    _CLIP_PREPROCESS = preprocess
    _CLIP_TOKENIZER = tokenizer
    _CLIP_TEXT_FEATURES = text_features

    return _CLIP_MODEL, _CLIP_PREPROCESS, _CLIP_TOKENIZER, _CLIP_TEXT_FEATURES


def get_opennsfw2():
    global _NSFW2, _NSFW2_ERROR
    if _NSFW2 is not None:
        return _NSFW2, None
    if _NSFW2_ERROR is not None:
        return None, _NSFW2_ERROR

    try:
        import opennsfw2
        _NSFW2 = opennsfw2
        return _NSFW2, None
    except Exception as e:
        _NSFW2_ERROR = f"opennsfw2 not available: {e}"
        return None, _NSFW2_ERROR


def initialize_feature_models(verbose: bool = True):
    """
    Optional one-time warmup for notebook use.
    """
    status = {}

    try:
        get_torch()
        status["device"] = get_device()
    except Exception as e:
        status["device_error"] = str(e)

    try:
        get_yolo_model()
        status["yolo"] = "ok"
    except Exception as e:
        status["yolo"] = f"error: {e}"

    try:
        get_easyocr_reader()
        status["easyocr"] = "ok"
    except Exception as e:
        status["easyocr"] = f"error: {e}"

    detector, err = get_mediapipe_face_detector()
    status["mediapipe_face"] = "ok" if detector is not None else f"error: {err}"

    try:
        get_clip_components()
        status["clip"] = "ok"
    except Exception as e:
        status["clip"] = f"error: {e}"

    nsfw2, err = get_opennsfw2()
    status["opennsfw2"] = "ok" if nsfw2 is not None else f"error: {err}"

    if verbose:
        for k, v in status.items():
            print(f"{k}: {v}")

    return status


# ----------------------------
# Feature extractors
# ----------------------------
def color_features(
    rgb: np.ndarray,
    palette_k: int = 5,
    sample_pixels: int = 20000,
    quant_bins: int = 32,
) -> Dict[str, Any]:
    hsv = rgb_to_hsv(rgb)
    Y = luminance(rgb)

    brightness = float(np.mean(Y) / 255.0)
    contrast = float(np.std(Y) / 255.0)
    saturation = float(np.mean(hsv[..., 1].astype(np.float32)) / 255.0)

    hue = hsv[..., 0].astype(np.float32)
    warm = np.logical_or(hue <= 20, hue >= 160)
    cool = np.logical_and(hue >= 80, hue <= 140)
    warm_ratio = float(np.mean(warm))
    cool_ratio = float(np.mean(cool))
    warmth_score = float(warm_ratio - cool_ratio)

    q = np.clip((rgb.astype(np.int32) * quant_bins) // 256, 0, quant_bins - 1)
    ids = q[..., 0] * (quant_bins * quant_bins) + q[..., 1] * quant_bins + q[..., 2]
    hist = np.bincount(ids.reshape(-1), minlength=quant_bins**3).astype(np.float32)
    p = hist / (hist.sum() + 1e-9)
    p_nz = p[p > 0]
    color_entropy = float(-np.sum(p_nz * np.log(p_nz + 1e-12)))

    dominant_color_count = int(np.sum(p > (1.0 / (quant_bins**3)) * 50.0))

    neon = np.logical_and(hsv[..., 1] >= 200, hsv[..., 2] >= 200)
    neon_ratio = float(np.mean(neon))

    skin = (
        (np.logical_or(hsv[..., 0] <= 25, hsv[..., 0] >= 160))
        & (hsv[..., 1] >= 40)
        & (hsv[..., 1] <= 200)
        & (hsv[..., 2] >= 60)
    )
    skin_tone_ratio = float(np.mean(skin))

    flat = rgb.reshape(-1, 3)
    n = flat.shape[0]
    if n > sample_pixels:
        idx = np.random.choice(n, size=sample_pixels, replace=False)
        samp = flat[idx]
    else:
        samp = flat

    km = KMeans(n_clusters=palette_k, n_init="auto", random_state=0)
    km.fit(samp.astype(np.float32))
    centers = km.cluster_centers_.astype(np.float32)
    counts = np.bincount(km.labels_, minlength=palette_k).astype(np.float32)
    order = np.argsort(-counts)
    centers = centers[order]
    counts = counts[order]
    weights = (counts / (counts.sum() + 1e-9)).tolist()

    palette = [
        {"rgb": centers[i].round(1).tolist(), "weight": float(weights[i])}
        for i in range(palette_k)
    ]

    return {
        "brightness": brightness,
        "contrast": contrast,
        "saturation": saturation,
        "warm_ratio": warm_ratio,
        "cool_ratio": cool_ratio,
        "warmth_score": warmth_score,
        "color_entropy": color_entropy,
        "dominant_color_count": dominant_color_count,
        "neon_ratio": neon_ratio,
        "skin_tone_ratio": skin_tone_ratio,
        "dominant_palette": palette,
    }


def edge_texture_layout_features(rgb: np.ndarray) -> Dict[str, Any]:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    edges = cv2.Canny(gray, threshold1=80, threshold2=160)
    edge_density = float(np.mean(edges > 0))

    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11 + 1), density=True)
    lbp_entropy = float(-np.sum(lbp_hist * np.log(lbp_hist + 1e-9)))

    mid = w // 2
    left = gray[:, :mid].astype(np.float32)
    right = gray[:, w - mid:].astype(np.float32)
    right_m = np.fliplr(right)
    mae = float(np.mean(np.abs(left - right_m)) / 255.0)
    symmetry_score = float(1.0 - mae)

    mag = cv2.Laplacian(gray, cv2.CV_32F)
    wts = np.abs(mag)
    wts_sum = float(wts.sum() + 1e-9)
    ys, xs = np.indices(gray.shape)
    visual_center_y = float((ys * wts).sum() / wts_sum) / float(h)
    visual_center_x = float((xs * wts).sum() / wts_sum) / float(w)

    left_mass = float(wts[:, :mid].sum())
    right_mass = float(wts[:, w - mid:].sum())
    visual_balance_lr = float(abs(left_mass - right_mass) / (left_mass + right_mass + 1e-9))

    midy = h // 2
    top_mass = float(wts[:midy, :].sum())
    bot_mass = float(wts[h - midy:, :].sum())
    top_heavy = float((top_mass - bot_mass) / (top_mass + bot_mass + 1e-9))
    bottom_heavy = float(-top_heavy)

    thirds = [(1/3, 1/3), (2/3, 1/3), (1/3, 2/3), (2/3, 2/3)]
    dists = [math.sqrt((visual_center_x - tx) ** 2 + (visual_center_y - ty) ** 2) for tx, ty in thirds]
    dmin = min(dists)
    rule_of_thirds_score = float(1.0 - clamp01(dmin / math.sqrt(2.0)))

    return {
        "edge_density": edge_density,
        "lbp_entropy": lbp_entropy,
        "symmetry_score": symmetry_score,
        "visual_center_x": visual_center_x,
        "visual_center_y": visual_center_y,
        "visual_balance_lr": visual_balance_lr,
        "top_heavy": top_heavy,
        "bottom_heavy": bottom_heavy,
        "rule_of_thirds_score": rule_of_thirds_score,
    }


def lighting_blur_negative_space_features(rgb: np.ndarray) -> Dict[str, Any]:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    Y = gray.astype(np.float32) / 255.0

    shadow_ratio = float(np.mean(Y < 0.18))
    highlight_ratio = float(np.mean(Y > 0.85))

    mu = float(np.mean(Y))
    sigma = float(np.std(Y) + 1e-9)
    luminance_skew = float(np.mean(((Y - mu) / sigma) ** 3))

    lap = cv2.Laplacian(gray, cv2.CV_32F)
    blur_score = float(lap.var())
    blur_score_norm = float(1.0 - math.exp(-blur_score / 200.0))

    g = cv2.GaussianBlur(gray, (5, 5), 0)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    grad = cv2.magnitude(gx, gy)
    thr = float(np.percentile(grad, 60.0))
    negative_space_ratio = float(np.mean(grad <= thr))

    return {
        "shadow_ratio": shadow_ratio,
        "highlight_ratio": highlight_ratio,
        "luminance_skew": luminance_skew,
        "blur_score": blur_score,
        "blur_score_norm": blur_score_norm,
        "negative_space_ratio": negative_space_ratio,
    }


def face_features_mediapipe(rgb: np.ndarray) -> Dict[str, Any]:
    detector, err = get_mediapipe_face_detector()
    if detector is None:
        return {
            "face_count": None,
            "largest_face_ratio": None,
            "avg_face_ratio": None,
            "face_centered_score": None,
            "face_error": err,
        }

    try:
        mp_image = _MP.Image(image_format=_MP.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)
    except Exception as e:
        return {
            "face_count": None,
            "largest_face_ratio": None,
            "avg_face_ratio": None,
            "face_centered_score": None,
            "face_error": f"mediapipe face model issue: {e}",
        }

    h, w, _ = rgb.shape
    boxes: List[Tuple[float, float, float, float]] = []
    for det in result.detections:
        bb = det.bounding_box
        boxes.append((bb.origin_x, bb.origin_y, bb.width, bb.height))

    face_count = int(len(boxes))
    if face_count == 0:
        return {
            "face_count": 0,
            "largest_face_ratio": 0.0,
            "avg_face_ratio": 0.0,
            "face_centered_score": 0.0,
        }

    areas = [max(0.0, bw) * max(0.0, bh) for _, _, bw, bh in boxes]
    largest_face_ratio = float(max(areas) / float(h * w))
    avg_face_ratio = float(np.mean(areas) / float(h * w))

    cx_img, cy_img = w / 2.0, h / 2.0
    dists = []
    for x, y, bw, bh in boxes:
        cx = x + bw / 2.0
        cy = y + bh / 2.0
        d = ((cx - cx_img) ** 2 + (cy - cy_img) ** 2) ** 0.5
        dists.append(d)

    dmin = min(dists)
    dmax = (cx_img**2 + cy_img**2) ** 0.5
    face_centered_score = float(1.0 - max(0.0, min(1.0, dmin / (dmax + 1e-9))))

    return {
        "face_count": face_count,
        "largest_face_ratio": largest_face_ratio,
        "avg_face_ratio": avg_face_ratio,
        "face_centered_score": face_centered_score,
    }


def object_features_yolov8(rgb: np.ndarray) -> Dict[str, Any]:
    try:
        torch = get_torch()
        model = get_yolo_model()
    except Exception as e:
        return {
            "object_count": None,
            "object_topk": None,
            "object_density": None,
            "person_count": None,
            "vehicle_count": None,
            "object_flags": None,
            "object_error": f"ultralytics not available: {e}",
        }

    try:
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        device_arg = 0 if torch.cuda.is_available() else "cpu"
        pred = model.predict(source=bgr, verbose=False, conf=0.25, device=device_arg)[0]
    except Exception as e:
        return {
            "object_count": None,
            "object_topk": None,
            "object_density": None,
            "person_count": None,
            "vehicle_count": None,
            "object_flags": None,
            "object_error": f"yolo inference failed: {e}",
        }

    names = pred.names
    if pred.boxes is None or pred.boxes.cls is None:
        det_names: List[str] = []
    else:
        cls = pred.boxes.cls.detach().cpu().numpy().astype(int)
        det_names = [names[i] for i in cls]

    h, w, _ = rgb.shape
    object_count = int(len(det_names))
    object_density = float(object_count / float(h * w) * 1e5)

    person_count = int(sum(n == "person" for n in det_names))
    vehicle_set = {"car", "bus", "truck", "motorcycle", "bicycle"}
    vehicle_count = int(sum(n in vehicle_set for n in det_names))

    flags = {
        "has_person": person_count > 0,
        "has_vehicle": vehicle_count > 0,
        "has_animal": any(n in {"dog", "cat", "horse", "bird", "bear"} for n in det_names),
    }

    return {
        "object_count": object_count,
        "object_topk": det_names[:10],
        "object_density": object_density,
        "person_count": person_count,
        "vehicle_count": vehicle_count,
        "object_flags": flags,
    }


def ocr_features_easyocr(rgb: np.ndarray) -> Dict[str, Any]:
    try:
        reader = get_easyocr_reader()
    except Exception as e:
        return {
            "ocr_text": None,
            "text_area_ratio": None,
            "uppercase_ratio": None,
            "num_text_boxes": None,
            "largest_text_area_ratio": None,
            "title_position_y": None,
            "stroke_width_mean": None,
            "stroke_width_std": None,
            "serifness_proxy": None,
            "ocr_error": f"easyocr not available: {e}",
        }

    try:
        result = reader.readtext(rgb)
    except Exception as e:
        return {
            "ocr_text": None,
            "text_area_ratio": None,
            "uppercase_ratio": None,
            "num_text_boxes": None,
            "largest_text_area_ratio": None,
            "title_position_y": None,
            "stroke_width_mean": None,
            "stroke_width_std": None,
            "serifness_proxy": None,
            "ocr_error": f"easyocr inference failed: {e}",
        }

    h, w, _ = rgb.shape
    total_area = float(h * w)
    text_area = 0.0
    texts: List[str] = []
    areas: List[float] = []
    centers_y: List[float] = []
    boxes_pts: List[np.ndarray] = []

    for bbox, text, conf in result:
        pts = np.array(bbox, dtype=np.float32)
        x_min, y_min = float(pts[:, 0].min()), float(pts[:, 1].min())
        x_max, y_max = float(pts[:, 0].max()), float(pts[:, 1].max())
        area = max(0.0, x_max - x_min) * max(0.0, y_max - y_min)
        cy = (y_min + y_max) / 2.0

        text_area += area
        texts.append(str(text))
        areas.append(area)
        centers_y.append(cy)
        boxes_pts.append(pts)

    joined = " ".join(texts).strip()
    alpha = [c for c in joined if c.isalpha()]
    uppercase = [c for c in alpha if c.isupper()]
    uppercase_ratio = safe_div(len(uppercase), len(alpha))

    num_text_boxes = int(len(result))
    text_area_ratio = float(text_area / (total_area + 1e-9))

    if len(areas) == 0:
        largest_text_area_ratio = 0.0
        title_position_y = 0.0
        stroke_width_mean = 0.0
        stroke_width_std = 0.0
        serifness_proxy = 0.0
    else:
        imax = int(np.argmax(areas))
        largest_text_area_ratio = float(areas[imax] / (total_area + 1e-9))
        title_position_y = float(centers_y[imax] / float(h))

        pts = boxes_pts[imax].astype(np.int32)
        x_min = int(np.clip(pts[:, 0].min(), 0, w - 1))
        x_max = int(np.clip(pts[:, 0].max(), 0, w - 1))
        y_min = int(np.clip(pts[:, 1].min(), 0, h - 1))
        y_max = int(np.clip(pts[:, 1].max(), 0, h - 1))

        crop = rgb[y_min:y_max + 1, x_min:x_max + 1]
        if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
            stroke_width_mean = 0.0
            stroke_width_std = 0.0
            serifness_proxy = 0.0
        else:
            gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
            bw = cv2.adaptiveThreshold(
                gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 10
            )
            dist = cv2.distanceTransform(bw, cv2.DIST_L2, 3)
            vals = dist[dist > 0]
            if vals.size == 0:
                stroke_width_mean = 0.0
                stroke_width_std = 0.0
                serifness_proxy = 0.0
            else:
                sw = 2.0 * vals
                stroke_width_mean = float(np.mean(sw))
                stroke_width_std = float(np.std(sw))

                neigh = cv2.filter2D((bw > 0).astype(np.uint8), -1, np.ones((3, 3), np.uint8))
                endpoints = np.logical_and(bw > 0, neigh == 2)
                serifness_proxy = float(np.mean(endpoints))

    return {
        "ocr_text": joined[:500],
        "text_area_ratio": text_area_ratio,
        "uppercase_ratio": float(uppercase_ratio),
        "num_text_boxes": num_text_boxes,
        "largest_text_area_ratio": largest_text_area_ratio,
        "title_position_y": title_position_y,
        "stroke_width_mean": float(stroke_width_mean),
        "stroke_width_std": float(stroke_width_std),
        "serifness_proxy": float(serifness_proxy),
    }


def nsfw_features_open_nsfw2(path: str) -> Dict[str, Any]:
    nsfw2, err = get_opennsfw2()
    if nsfw2 is None:
        return {"nsfw_score": None, "nsfw_error": err}

    try:
        score = float(nsfw2.predict_image(path))
        return {"nsfw_score": score}
    except Exception as e:
        return {"nsfw_score": None, "nsfw_error": f"opennsfw2 failed: {e}"}


def clip_features_openclip(rgb: np.ndarray) -> Dict[str, Any]:
    try:
        torch = get_torch()
        device = get_device()
        model, preprocess, tokenizer, text_features = get_clip_components()
    except Exception as e:
        return {
            "clip_embedding": None,
            "clip_prompt_scores": None,
            "clip_model": None,
            "clip_error": f"open_clip/torch not available: {e}",
        }

    try:
        pil = Image.fromarray(rgb)
        image_tensor = preprocess(pil).unsqueeze(0).to(device)

        with torch.no_grad():
            img_feat = model.encode_image(image_tensor)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        sims = (img_feat @ text_features.T).squeeze(0).detach().cpu().numpy()
        clip_prompt_scores = {k: float(v) for k, v in zip(CLIP_PROMPTS.keys(), sims)}

        return {
            "clip_embedding": img_feat.squeeze(0).detach().cpu().numpy().astype(np.float32),
            "clip_prompt_scores": clip_prompt_scores,
            "clip_model": {"model": CLIP_MODEL_NAME, "pretrained": CLIP_PRETRAINED},
        }
    except Exception as e:
        return {
            "clip_embedding": None,
            "clip_prompt_scores": None,
            "clip_model": None,
            "clip_error": f"clip inference failed: {e}",
        }


# ----------------------------
# One-call extractor
# ----------------------------
def extract_poster_nodes(path: str) -> Dict[str, Any]:
    rgb = load_rgb(path, max_side=MAX_SIDE)

    nodes: Dict[str, Any] = {
        "path": path,
        "width": int(rgb.shape[1]),
        "height": int(rgb.shape[0]),
    }

    nodes.update(color_features(rgb))
    nodes.update(edge_texture_layout_features(rgb))
    nodes.update(lighting_blur_negative_space_features(rgb))
    nodes.update(face_features_mediapipe(rgb))
    nodes.update(object_features_yolov8(rgb))
    nodes.update(ocr_features_easyocr(rgb))
    nodes.update(clip_features_openclip(rgb))
    nodes.update(nsfw_features_open_nsfw2(path))

    return nodes

In [43]:
#Example usage

In [45]:
# import json

# poster_path = "eeaaoposter.jpg"
# nodes = extract_poster_nodes(poster_path)

# nodes_print = dict(nodes)
# if isinstance(nodes_print.get("clip_embedding"), np.ndarray):
#     emb = nodes_print["clip_embedding"]
#     nodes_print["clip_embedding"] = {
#         "dim": int(emb.shape[0]),
#         "l2_norm": float(np.linalg.norm(emb)),
#         "preview": emb[:8].round(4).tolist(),
#     }

# print(json.dumps(nodes_print, indent=2))